# 🇧🇩 Bangla Handwritten Character Recognition (SE-ResNet + RL Refinement)

This notebook provides a complete 2-stage training & evaluation pipeline:
1. **Stage 1: SE-ResNet Classification Training** (CrossEntropyLoss, Label Smoothing, AdamW, Cosine Annealing)
2. **Stage 2: Reinforcement Learning (REINFORCE) Refinement** (Policy Gradient fine-tuning on hard/confused characters)
3. **Evaluation & Visualization** (Per-Class Accuracy, Confusion Analysis, Inference Testing)

In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import classification_report, confusion_matrix

from model import BestCNN
from dataset import prepare_data, load_images
from rl_agent import run_rl_refinement
from train import get_device, train_one_epoch, validate

device = get_device()
print(f"Using PyTorch device: {device}")

## Step 1: Load Dataset & Data Loaders

In [ ]:
train_loader, val_loader, le, num_classes = prepare_data(batch_size=128)
print(f"Total Character Classes: {num_classes}")

## Step 2: Initialize SE-ResNet Architecture

In [ ]:
model = BestCNN(num_classes).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"BestCNN Initialized with {total_params:,} parameters")

## Step 3: Stage 1 — Classification Pre-Training

In [ ]:
EPOCHS = 50
LR = 0.0005
PATIENCE = 15

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)

train_losses, val_losses, val_accuracies = [], [], []
best_val_acc = 0.0
best_state = None
patience_cnt = 0

print("Starting Stage 1 Training...")
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    scheduler.step()
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)
    
    marker = ""
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_cnt = 0
        marker = " ★ BEST"
    else:
        patience_cnt += 1
    
    print(f"Epoch {epoch:02d}/{EPOCHS} | Train Loss: {train_loss:.4f} Acc: {train_acc:.1f}% | Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}%{marker}")
    if patience_cnt >= PATIENCE:
        print(f"Early stopping triggered at epoch {epoch}")
        break

if best_state:
    model.load_state_dict(best_state)
print(f"Stage 1 Complete! Best Validation Accuracy: {best_val_acc:.2f}%")

## Step 4: Stage 2 — Reinforcement Learning (REINFORCE) Refinement

In [ ]:
model = run_rl_refinement(model, train_loader, val_loader, device, episodes=10, lr=1e-4)
torch.save(model.state_dict(), "checkpoints/best_cnn_model_weights.pth")
torch.save(model.state_dict(), "best_cnn_model_weights.pth")
print("✓ Saved final weights!")

## Step 5: Visual Evaluation & Performance Metrics

In [ ]:
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.title("Loss Curves")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(val_accuracies, label="Val Accuracy", color="green")
plt.axhline(best_val_acc, color="red", linestyle="--", label=f"Best: {best_val_acc:.2f}%")
plt.title("Validation Accuracy (%)")
plt.xlabel("Epoch")
plt.legend()
plt.tight_layout()
plt.show()